# Практическая работа 3. Локальный запуск моделей

Выполнены задания:

2. Влияние квантизации в llama.cpp.
3. CPU и GPU.

Используется модель SmolLM2-135M-Instruct и библиотека `llama-cpp-python` (обертка над llama.cpp). Модель маленькая, поэтому все замеры можно провести на обычном ноутбуке. Запросы к модели написаны на английском, так как русский язык она понимает плохо.

In [1]:
# pip install llama-cpp-python llm-smollm2 psutil pandas matplotlib

In [2]:
import os, time, ctypes, importlib.util, threading
import psutil
import pandas as pd
import llama_cpp
from llama_cpp import Llama

# отключение служебных сообщений llama.cpp
@llama_cpp.llama_log_callback
def quiet_log(level, text, user_data):
    pass
llama_cpp.llama_log_set(quiet_log, ctypes.c_void_p())

SOURCE_MODEL = os.path.join(importlib.util.find_spec("llm_smollm2").submodule_search_locations[0],
                            "SmolLM2-135M-Instruct.Q4_1.gguf")
process = psutil.Process()
print("Ядер процессора:", os.cpu_count(), "| RAM, ГБ:", round(psutil.virtual_memory().total / 1e9, 1))

Ядер процессора: 4 | RAM, ГБ: 16.9


## Задание 2. Влияние квантизации

### 2.1. Получение версий Q4, Q5 и Q8
Модель распространяется в формате Q4_1. Из нее с помощью функции квантизации llama.cpp получены версии Q4_K_M, Q5_K_M и Q8_0. Так как исходник уже квантован, версии Q5 и Q8 не могут быть точнее исходной модели. Поэтому по качеству ожидаются небольшие различия, а по размеру и скорости различия будут видны.

In [3]:
os.makedirs("models", exist_ok=True)
VERSIONS = {"Q4_K_M": llama_cpp.LLAMA_FTYPE_MOSTLY_Q4_K_M,
            "Q5_K_M": llama_cpp.LLAMA_FTYPE_MOSTLY_Q5_K_M,
            "Q8_0": llama_cpp.LLAMA_FTYPE_MOSTLY_Q8_0}
paths = {}
for name, ftype in VERSIONS.items():
    params = llama_cpp.llama_model_quantize_default_params()
    params.ftype = ftype
    params.allow_requantize = True
    paths[name] = f"models/smollm2-135m-{name.lower()}.gguf"
    llama_cpp.llama_model_quantize(SOURCE_MODEL.encode(), paths[name].encode(), ctypes.byref(params))
    print(name, round(os.path.getsize(paths[name]) / 2**20, 1), "МБ")

Q4_K_M 100.6 МБ


Q5_K_M 106.9 МБ


Q8_0 138.1 МБ


### 2.2. Время загрузки, память и скорость генерации
Для каждой версии: загрузка модели (3 раза), прирост памяти процесса после загрузки и скорость генерации 128 токенов (3 раза). Модель загружается без `mmap`, чтобы вся она сразу попадала в память и замер RAM был честным.

In [4]:
def load(path, **kw):
    return Llama(path, n_ctx=2048, use_mmap=False, verbose=False, **kw)

def generate(llm, prompt, max_tokens=128):
    t = time.perf_counter()
    r = llm.create_chat_completion(messages=[{"role": "user", "content": prompt}],
                                   max_tokens=max_tokens, temperature=0, seed=0)
    sec = time.perf_counter() - t
    return r["choices"][0]["message"]["content"], r["usage"], sec

SPEED_PROMPT = "Write a short story about a cat who learns to cook."

rows = []
for name, path in paths.items():
    load_times = []
    for _ in range(3):
        t = time.perf_counter(); llm = load(path); load_times.append(time.perf_counter() - t); del llm
    ram_before = process.memory_info().rss
    llm = load(path)
    ram_mb = (process.memory_info().rss - ram_before) / 2**20
    speeds = []
    for _ in range(3):
        _, usage, sec = generate(llm, SPEED_PROMPT)
        speeds.append(usage["completion_tokens"] / sec)
    rows.append({"версия": name, "размер, МБ": round(os.path.getsize(path) / 2**20, 1),
                 "загрузка, с": round(sum(load_times) / 3, 3), "RAM, МБ": round(ram_mb),
                 "токенов/с": round(sum(speeds) / 3, 1)})
    del llm

perf = pd.DataFrame(rows)
perf

,версия,"размер, МБ","загрузка, с","RAM, МБ",токенов/с
0,Q4_K_M,100.6,0.203,136,101.2
1,Q5_K_M,106.9,0.186,150,86.2
2,Q8_0,138.1,0.197,181,78.3


### 2.3. Длинный запрос (500+ токенов)

In [5]:
LONG_TEXT = " ".join([
    "The city library opened in 1925 and quickly became the center of public life.",
    "At first it had only two rooms and about three thousand books, most of them donated by local families.",
    "In 1958 a new building was constructed near the central square, with a large reading hall and a children's department.",
    "During the 1970s the library started a program of evening lectures about science, history and art.",
    "The lectures were free and attracted students, workers and retired people.",
    "In 1994 the first computers with internet access appeared in the reading hall, and the library began to digitize old newspapers.",
    "Today the collection contains more than two hundred thousand items, including books, maps, photographs and audio recordings.",
    "Every summer the library organizes a reading marathon for children, and the winners receive small prizes.",
    "Volunteers help elderly visitors learn how to use smartphones and online services.",
    "The library also has a small cafe and a garden where people can read outdoors in warm weather.",
] * 5)
LONG_PROMPT = "Summarize the following text in three sentences.\n\n" + LONG_TEXT

rows = []
for name, path in paths.items():
    llm = load(path)
    n_in = len(llm.tokenize(LONG_PROMPT.encode()))
    answer, usage, sec = generate(llm, LONG_PROMPT, max_tokens=150)
    rows.append({"версия": name, "токенов на входе": n_in, "токенов ответа": usage["completion_tokens"],
                 "время ответа, с": round(sec, 2), "начало ответа": answer[:120].replace("\n", " ")})
    del llm
pd.DataFrame(rows)

,версия,токенов на входе,токенов ответа,"время ответа, с",начало ответа
0,Q4_K_M,1027,106,4.38,"The city library opened its doors in 1925, qui..."
1,Q5_K_M,1027,150,5.56,The city library has been a beacon of public l...
2,Q8_0,1027,102,4.54,The city library has been a cornerstone of pub...


### 2.4. Качество ответов
Каждой версии даны три задачи: логическая, генерация кода и пересказ текста.

In [6]:
TASKS = {
    "логика": "Tom is taller than Anna. Anna is taller than Mike. Who is the shortest? Answer with one name.",
    "код": "Write a Python function that returns the sum of even numbers in a list.",
    "пересказ": "Retell this text in one sentence: " + LONG_TEXT[:600],
}
answers = {}
for name, path in paths.items():
    llm = load(path)
    for task, prompt in TASKS.items():
        answers[(name, task)] = generate(llm, prompt, max_tokens=150)[0]
    del llm

for task in TASKS:
    print("=" * 30, task)
    for name in paths:
        print(f"--- {name}:\n{answers[(name, task)]}\n")

============================== логика
--- Q4_K_M:
The shortest person in the world is Emily.

--- Q5_K_M:
The shortest is actually Anna, who is taller than Mike.

--- Q8_0:
The shortest person in this scenario is Mike.

============================== код
--- Q4_K_M:
```python
def sum_even_numbers(numbers):
    return sum(num for num in numbers if isinstance(num, (int, float)) and isinstance(num, (int, float)))
```

This function uses a generator expression to create a list of even numbers and then uses the built-in `sum` function to add them up. The `isinstance` function is used to check if the input is an instance of the `int` and `float` types. If it is, it is added to the sum.

--- Q5_K_M:
```python
def sum_even_numbers(numbers):
    return sum(num for num in numbers if isinstance(num, (int, float)) and isinstance(num, (int, float)))
```

This function uses a generator expression to iterate over the input list and only include numbers that are both integers and floats. The `isinstan

Оценка ответов:

| Задача | Q4_K_M | Q5_K_M | Q8_0 |
|---|---|---|---|
| Логика (правильный ответ: Mike) | неверно, придумано имя | неверно | верно |
| Код | функция не проверяет четность, ошибка | та же ошибка | функция верная, но в примере неверный результат (15 вместо 6) |
| Пересказ | в целом верно, но больше одного предложения и есть неточность про комнаты | фактическая ошибка (компьютер в 1925 году) | верно и кратко |

Лучшие ответы дала версия Q8_0. Версии Q4_K_M и Q5_K_M при повторном квантовании получили дополнительную ошибку округления весов, и для такой маленькой модели это заметно.

**Вывод по заданию 2.** Чем меньше бит на вес, тем меньше файл и расход памяти: Q4_K_M занимает около 100 МБ, Q8_0 около 138 МБ, разница по RAM примерно такая же. Время загрузки у всех версий меньше 0,3 секунды и почти не отличается. Скорость генерации выше всего у Q4_K_M (около 100 токенов/с), у Q5_K_M около 86, у Q8_0 около 78 токенов/с. Замеры скорости немного меняются от запуска к запуску, но порядок версий сохраняется. По качеству лучшей оказалась Q8_0. Для маленьких моделей экономия от Q4 небольшая, а потеря качества заметная, поэтому лучше брать Q8_0. Для больших моделей (7B и больше), где важна экономия памяти, разумный компромисс дают Q4_K_M и Q5_K_M.

## Задание 3. CPU и GPU

### 3.1. Проверка GPU

In [7]:
print("Поддержка GPU в llama.cpp:", llama_cpp.llama_supports_gpu_offload())

Поддержка GPU в llama.cpp: False


На компьютере, где выполнялась работа, нет видеокарты с поддержкой CUDA, а `llama-cpp-python` собран без поддержки GPU. Поэтому часть задания с GPU не выполнена. Для запуска на GPU нужно установить версию библиотеки с CUDA и передать при загрузке параметр `n_gpu_layers=-1` (все слои модели переносятся на видеокарту). Ниже подробно исследована работа на CPU.

### 3.2. Скорость на CPU, загрузка процессора и память
Замеры сделаны для разного числа потоков. Время до первого токена измеряется в потоковом режиме (`stream=True`). Загрузка CPU считывается в отдельном потоке во время генерации.

In [8]:
def run_stream(llm, prompt, max_tokens=128):
    t = time.perf_counter(); first = None; n = 0
    for chunk in llm.create_chat_completion(messages=[{"role": "user", "content": prompt}],
                                            max_tokens=max_tokens, temperature=0, seed=0, stream=True):
        if chunk["choices"][0]["delta"].get("content"):
            n += 1
            if first is None:
                first = time.perf_counter() - t
    return first, n, time.perf_counter() - t

def cpu_load_during(func):
    samples, done = [], threading.Event()
    def watch():
        psutil.cpu_percent(None)
        while not done.is_set():
            time.sleep(0.1); samples.append(psutil.cpu_percent(None))
    th = threading.Thread(target=watch); th.start()
    result = func(); done.set(); th.join()
    return result, sum(samples) / max(len(samples), 1)

MODEL = paths["Q4_K_M"]
rows = []
for threads in [1, 2, 4]:
    llm = load(MODEL, n_threads=threads)
    run_stream(llm, SPEED_PROMPT, 8)    # прогрев
    (first, n, total), cpu = cpu_load_during(lambda: run_stream(llm, SPEED_PROMPT))
    rows.append({"потоков": threads, "до первого токена, с": round(first, 3), "токенов/с": round(n / total, 1),
                 "загрузка CPU, %": round(cpu), "RAM процесса, МБ": round(process.memory_info().rss / 2**20)})
    del llm
cpu_df = pd.DataFrame(rows)
cpu_df

,потоков,"до первого токена, с",токенов/с,"загрузка CPU, %","RAM процесса, МБ"
0,1,0.024,50.0,25,365
1,2,0.013,122.6,48,365
2,4,0.005,174.9,91,365


### 3.3. Стресс-тест
10 последовательных запросов к одной модели и 5 параллельных запросов (каждый поток работает со своей копией модели, так как один объект `Llama` нельзя использовать из нескольких потоков одновременно).

In [9]:
QUESTIONS = ["What is the capital of France?", "Name three colors.", "What is 2 + 2?",
             "Why is the sky blue?", "Give a synonym for happy.", "What do cats eat?",
             "Name a planet.", "What is water made of?", "How many days are in a week?",
             "What is a computer?"]

llm = load(MODEL)
seq = []
for q in QUESTIONS:
    _, _, sec = generate(llm, q, max_tokens=64)
    seq.append(sec)
del llm

models = [load(MODEL, n_threads=1) for _ in range(5)]
par = [None] * 5
def worker(i):
    par[i] = generate(models[i], QUESTIONS[i], max_tokens=64)[2]
t = time.perf_counter()
threads = [threading.Thread(target=worker, args=(i,)) for i in range(5)]
for th in threads: th.start()
for th in threads: th.join()
par_total = time.perf_counter() - t
del models

stress = pd.DataFrame([
    {"режим": "10 последовательных", "средняя задержка, с": round(sum(seq) / len(seq), 2),
     "мин, с": round(min(seq), 2), "макс, с": round(max(seq), 2), "общее время, с": round(sum(seq), 2)},
    {"режим": "5 параллельных", "средняя задержка, с": round(sum(par) / len(par), 2),
     "мин, с": round(min(par), 2), "макс, с": round(max(par), 2), "общее время, с": round(par_total, 2)},
])
stress

,режим,"средняя задержка, с","мин, с","макс, с","общее время, с"
0,10 последовательных,0.44,0.13,0.95,4.45
1,5 параллельных,1.15,0.51,1.97,1.97


При 5 параллельных запросах средняя задержка одного запроса выросла примерно в 2,5 раза (с 0,44 до 1,15 с), так как потоки делят 4 ядра процессора. При этом 5 ответов получены за 2 секунды, то есть общая пропускная способность немного выше, чем при последовательной обработке. Длина ответов разная, поэтому разброс задержек большой (от 0,13 до 1,97 с).

**Вывод по заданию 3.** На CPU скорость растет с числом потоков: 1 поток дает около 50 токенов/с, 4 потока около 175 токенов/с, загрузка процессора при этом доходит до 91%. Память от числа потоков не зависит. По умолчанию `llama-cpp-python` использует половину ядер (здесь 2 потока), поэтому в задании 2 скорость была ниже максимальной. Для маленькой модели CPU вполне достаточно: время до первого токена меньше 0,03 с. GPU оправдан для моделей от нескольких миллиардов параметров и для обработки многих запросов одновременно: на CPU параллельные запросы конкурируют за ядра, и задержка каждого растет.